# Setting up / Resetting data stores and tables for Persuasio/PersuasUI

Save and reset contents of local/SQL datastores for Persuasio and PersuasUI

These operations just focus on contents of dbs. To restart the actual remote servers, see `Code/persuasui/deployment/aliases`

## Imports / Methods

In [ ]:
from typing import List
import os
import json
from datetime import datetime as dt
import psycopg2 as pg

import pandas as pd
pd.set_option('display.max_columns', None)

In [2]:
def create_table(sql, table: str, schema: str):
    """ 
    Create table in sql db using provided schema if it does not already exist.
    """
    cursor = sql.cursor()
    cursor.execute(f"CREATE TABLE IF NOT EXISTS {table} ({schema});")
    sql.commit()
    cursor.close()

def list_tables(sql) -> List[str]:
    """
    List all tables in the public schema of the sql db.
    """
    cursor = sql.cursor()
    cursor.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
    """)
    tables = [row[0] for row in cursor.fetchall()]
    cursor.close()
    return tables

def get_table(sql, table: str):
    """
    Retrieve all data from a table as a list of dicts.
    """
    cursor = sql.cursor()
    cursor.execute(f"SELECT * FROM {table};")
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    data = [dict(zip(columns, row)) for row in rows]
    cursor.close()
    return data

def save_table(table_data: List[dict], table_name: str, data_path: str):
    """
    Save each table as a JSON file in the specified data path.
    """
    with open(os.path.join(data_path, f"{table_name}.json"), 'w') as f:
        json.dump(table_data, f, indent=4, default=str)
        
def get_table_size(sql, table: str) -> int:
    """
    Get the number of rows in a table.
    """
    cursor = sql.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    size = cursor.fetchone()[0]
    cursor.close()
    return size

def list_tables_with_sizes(sql) -> List[tuple]:
    """
    List all tables with their row counts.
    """
    tables = list_tables(sql)
    return [(t, get_table_size(sql, t)) for t in tables]
        
def empty_table(sql, table: str):
    """
    Empty the specified table in the SQL database.
    """
    size = get_table_size(sql, table)
    if size == 0:
        print(f"Table {table} is already empty.")
        return
    cursor = sql.cursor()
    cursor.execute(f"DELETE FROM {table};")
    print(f"Table {table} emptied ({size} rows).")
    sql.commit()
    cursor.close()

def fill_table(sql, table: str, data: List[dict]):
    """
    Fill the specified table with data (list of dicts).
    Assumes the table schema matches the keys in the dicts.
    """
    cursor = sql.cursor()
    columns = data[0].keys()
    values_str = ', '.join(['%s'] * len(columns))
    insert_query = f"INSERT INTO {table} ({', '.join(columns)}) VALUES ({values_str});"
    for row in data:
        values = []
        for col in columns:
            val = row[col]
            # Convert dicts to JSON strings for jsonb columns
            if col.endswith('_data'):
                values.append(json.dumps(val))
            else:
                values.append(val)
        cursor.execute(insert_query, tuple(values))
    sql.commit()
    cursor.close()
    
def replace_table(sql, table: str, data: List[dict]):
    """
    Replace the contents of a table with new data.
    """
    empty_table(sql, table)
    fill_table(sql, table, data)
        
    
def download_and_empty_all_tables(sql, prefix: str, data_dir: str = "backups", skip_empty: bool = True):
    """
    Batch operation to download all tables as JSON files and then empty them.
    """
    now = dt.now().strftime("%Y%m%d_%H%M%S")
    data_path = f"{data_dir.strip('/')}/{prefix}_{now}"
    if not os.path.exists(data_path):
        os.makedirs(data_path)
    tables = list_tables(sql)
    for t in tables:
        table_data = get_table(sql, t)
        if skip_empty and len(table_data) == 0:
            print(f"Table {t} is empty, skipping.")
        else:
            save_table(table_data, t, data_path)
            print(f"Table {t} saved to {data_path}/{t}.json")
            empty_table(sql, t)
            print(f"Table {t} emptied.")
    print(f"All tables downloaded and emptied for {prefix}.")
    
def list_connections(conn):
    """List all connections to the database."""
    cur = conn.cursor()
    cur.execute("""
        SELECT pid, state, wait_event_type, query
        FROM pg_stat_activity
        WHERE datname = current_database()
        AND pid != pg_backend_pid()
    """)
    connections = cur.fetchall()
    cur.close()
    return connections

def kill_connections(conn):
    """Kill idle-in-transaction and blocked connections."""
    cur = conn.cursor()

    cur.execute("""
        SELECT pg_terminate_backend(pid)
        FROM pg_stat_activity
        WHERE datname = current_database()
        AND state = 'idle in transaction'
        AND pid != pg_backend_pid()
    """)
    idle_killed = cur.fetchall()
    conn.commit()

    cur.execute("""
        SELECT pg_terminate_backend(pid)
        FROM pg_stat_activity
        WHERE datname = current_database()
        AND wait_event_type = 'Lock'
        AND pid != pg_backend_pid()
    """)
    blocked_killed = cur.fetchall()
    conn.commit()

    cur.close()
    return {'idle_killed': len(idle_killed), 'blocked_killed': len(blocked_killed)}

## Connection Setup

References remote sql databases on aca-deployment, or local

In [8]:
REMOTE = True

if REMOTE:
    sm = pg.connect(
        dbname="session_manager",
        user="session_manager",
        password="dsoDBATILivLee0",
        host="psql-session-manager.postgres.database.azure.com",
        port=5432
    )

    per = pg.connect(
        dbname="persuasio",
        user="persuasio",
        password="dsoDBATILivLee0",
        host="psql-persuasio.postgres.database.azure.com",
        port=5432
    )

    logs = pg.connect(
        dbname="logs",
        user="logs",
        password="dsoDBATILivLee0",
        host="psql-logs.postgres.database.azure.com",
        port=5432
    )
else:
    sm = pg.connect(
        dbname="session_manager",
        user="postgres",
        password="password",
        host="localhost",
        port=5432
    )

    per = pg.connect(
        dbname="persuasio",
        user="postgres",
        password="password",
        host="localhost",
        port=5432
    )

    logs = pg.connect(
        dbname="logs",
        user="postgres",
        password="password",
        host="localhost",
        port=5432
    )

In [15]:
print(list_tables_with_sizes(per))
print(list_tables_with_sizes(sm))
print(list_tables_with_sizes(logs))

[('checkpoint_migrations', 0), ('checkpoints', 0), ('checkpoint_blobs', 0), ('checkpoint_writes', 0), ('session_data', 0), ('runtime_states', 0), ('final_states', 0), ('terminated_states', 0)]
[('sessions', 48), ('participants', 20), ('dialogues', 0)]
[('persuasio', 0), ('session_manager', 0), ('client', 0)]


In [16]:
get_table(sm, "participants")

[{'participant_id': 'human_1',
  'participant_data': {'is_admin': False,
   'auth_code': 'RWF1M0',
   'last_activity': None,
   'message_count': 0,
   'participant_id': 'human_1',
   'current_session': None,
   'is_authenticated': False,
   'participant_type': 'human'}},
 {'participant_id': 'human_2',
  'participant_data': {'is_admin': False,
   'auth_code': 'Y0ISVK',
   'last_activity': None,
   'message_count': 0,
   'participant_id': 'human_2',
   'current_session': None,
   'is_authenticated': False,
   'participant_type': 'human'}},
 {'participant_id': 'human_3',
  'participant_data': {'is_admin': False,
   'auth_code': 'XNR1JE',
   'last_activity': None,
   'message_count': 0,
   'participant_id': 'human_3',
   'current_session': None,
   'is_authenticated': False,
   'participant_type': 'human'}},
 {'participant_id': 'human_4',
  'participant_data': {'is_admin': False,
   'auth_code': '6EM83U',
   'last_activity': None,
   'message_count': 0,
   'participant_id': 'human_4',
   '

## Persuasio

- Need to empty tables inbetween runs
- May need to check/kill idle connections

In [4]:
list_tables_with_sizes(per)

[('checkpoint_migrations', 6),
 ('checkpoints', 0),
 ('checkpoint_blobs', 0),
 ('checkpoint_writes', 0),
 ('session_data', 0),
 ('runtime_states', 0),
 ('final_states', 0),
 ('terminated_states', 0)]

In [98]:
print(list_tables_with_sizes(per))

download_and_empty_all_tables(
    per,
    prefix="persuasio",
    data_dir="backups"
)

list_tables_with_sizes(per)

[('checkpoint_migrations', 10), ('checkpoints', 0), ('checkpoint_blobs', 0), ('checkpoint_writes', 0), ('session_data', 0), ('runtime_states', 0), ('final_states', 0), ('terminated_states', 0)]
Table checkpoint_migrations saved to backups/persuasio_20251209_105113/checkpoint_migrations.json
Table checkpoint_migrations emptied (10 rows).
Table checkpoint_migrations emptied.
Table checkpoints is empty, skipping.
Table checkpoint_blobs is empty, skipping.
Table checkpoint_writes is empty, skipping.
Table session_data is empty, skipping.
Table runtime_states is empty, skipping.
Table final_states is empty, skipping.
Table terminated_states is empty, skipping.
All tables downloaded and emptied for persuasio.


[('checkpoint_migrations', 0),
 ('checkpoints', 0),
 ('checkpoint_blobs', 0),
 ('checkpoint_writes', 0),
 ('session_data', 0),
 ('runtime_states', 0),
 ('final_states', 0),
 ('terminated_states', 0)]

In [111]:
# Check connections
print(list_connections(per))

# Kill connections (in case of issues)
# kill_connections(per)

[(53667, 'idle', 'Client', "\n            SELECT COUNT(*) FROM pg_extension WHERE extname IN (\n                'edb_job_scheduler', 'dbms_scheduler') ")]


## Session Manager

- Need to replace tables between runs (i.e. empty and fill with template data)

In [102]:
list_tables_with_sizes(sm)

[('sessions', 252), ('participants', 16), ('dialogues', 0)]

In [89]:
# empty and save dialogues

print(list_tables_with_sizes(sm))

download_and_empty_all_tables(sm, prefix="sessionmanager", data_dir="backups", skip_empty=False)

print(list_tables_with_sizes(sm))

[('sessions', 252), ('participants', 16), ('dialogues', 6)]
Table sessions saved to backups/sessionmanager_20251209_093817/sessions.json
Table sessions emptied (252 rows).
Table sessions emptied.
Table participants saved to backups/sessionmanager_20251209_093817/participants.json
Table participants emptied (16 rows).
Table participants emptied.
Table dialogues saved to backups/sessionmanager_20251209_093817/dialogues.json
Table dialogues emptied (6 rows).
Table dialogues emptied.
All tables downloaded and emptied for sessionmanager.
[('sessions', 0), ('participants', 0), ('dialogues', 0)]


In [101]:
# replace sessions and participants with session_manger/data/templates/{whatever template to use}

TEMPLATE = "full"
sessions = json.load(open(f"session_manager/data/templates/{TEMPLATE}/sessions.json", 'r'))
participants = json.load(open(f"session_manager/data/templates/{TEMPLATE}/participants.json", 'r'))

session_rows = [
    {'session_id': s['session_id'], 'session_data': s}
    for s in sessions
]

participant_rows = [
    {'participant_id': p['participant_id'], 'participant_data': p}
    for p in participants
]


if 'sessions' not in list_tables(sm):
    create_table(
        sm,
        table="sessions",
        schema="""
            session_id TEXT PRIMARY KEY,
            session_data JSONB
        """
    )
if 'participants' not in list_tables(sm):
    create_table(
        sm,
        table="participants",
        schema="""
            participant_id TEXT PRIMARY KEY,
            participant_data JSONB
        """
    )

replace_table(sm, 'sessions', session_rows)
replace_table(sm, 'participants', participant_rows)

list_tables_with_sizes(sm)

Table sessions emptied (252 rows).
Table participants emptied (16 rows).


[('sessions', 252), ('participants', 16), ('dialogues', 0)]

In [91]:
# test loading

from session_manager.session_manager.models import Session, SessionParameters, Participant

session_objects = [
    Session(**s['session_data']) for s in get_table(sm, 'sessions')
]
participant_objects = [
    Participant(**p['participant_data']) for p in get_table(sm, 'participants')
]

assert isinstance(session_objects[0], Session)
assert isinstance(session_objects[0].parameters, SessionParameters)

## Logs

- Need to empty logs between runs
- May need to check/kill idle connections

In [104]:
list_tables_with_sizes(logs)

[('session_manager', 3), ('client', 1), ('persuasio', 0)]

In [93]:
print(list_tables_with_sizes(logs))

download_and_empty_all_tables(
    logs,
    prefix="logs",
    data_dir="backups"
)

list_tables_with_sizes(logs)

[('persuasio', 245), ('session_manager', 44), ('client', 65)]
Table persuasio saved to backups/logs_20251209_093827/persuasio.json
Table persuasio emptied (245 rows).
Table persuasio emptied.
Table session_manager saved to backups/logs_20251209_093827/session_manager.json
Table session_manager emptied (44 rows).
Table session_manager emptied.
Table client saved to backups/logs_20251209_093827/client.json
Table client emptied (65 rows).
Table client emptied.
All tables downloaded and emptied for logs.


[('persuasio', 0), ('session_manager', 0), ('client', 0)]

In [110]:
# Check connections
print(list_connections(logs))

# Kill connections (in case of issues)
# kill_connections(logs)

[(53699, 'idle', 'Client', "\n            SELECT COUNT(*) FROM pg_extension WHERE extname IN (\n                'edb_job_scheduler', 'dbms_scheduler') ")]


In [ ]:
# make logs into pandas dfs for readability
smlogs = get_table(logs, 'session_manager')
smlogs_df = pd.DataFrame(smlogs)

cllogs = get_table(logs, 'client')
cllogs_df = pd.DataFrame(cllogs)

pelogs  = get_table(logs, 'persuasio')
pelogs_df = pd.DataFrame(pelogs)

In [106]:
pelogs_df

""


In [107]:
smlogs_df

,id,timestamp,event_type,severity,participant_id,message,session_id
0,70,2025-12-09 10:02:19.213689,system,info,SYSTEM,Session manager initialised,None
1,71,2025-12-09 10:02:19.280029,system,info,SYSTEM,Main database connection established and healthy,None
2,72,2025-12-09 10:02:19.280691,system,info,SYSTEM,Logs database connection established and healthy,None


In [108]:
cllogs_df

,id,timestamp,event_type,severity,participant_id,message,session_id
0,86,2025-12-09 10:02:20.257345,system,info,SYSTEM,Launching Gradio client on 0.0.0.0:7860,None
